# 04 — Récurrences et échéanciers

Les deux questions qui ont motivé la bibliothèque :

```python
last_weekday("2026-01-01", "2026-12-31", FRI)                # dernier vendredi de chaque mois
nth_weekday("2026-01-01", "2026-12-31", 2, THU, freq="Q")    # 2e jeudi de chaque trimestre
```

Puis `Schedule`, pour les échéanciers de coupons — avec la séparation stricte entre dates
**non ajustées** et dates **ajustées**, qui est la décision de conception la plus
structurante de tout le module.

In [1]:
from datetime import date

import pandas as pd

import better_calendar as bcal
from better_calendar import FRI, MON, SAT, SUN, THU, TUE, WED, Schedule, Weekday

def iso(index):
    return list(index.strftime("%Y-%m-%d"))

Les constantes de jour sont un `IntEnum` aligné sur `date.weekday()` — lundi vaut 0, donc
elles s'utilisent partout où la bibliothèque standard attend un indice de jour.

In [2]:
print("valeurs :", {j.name: int(j) for j in Weekday})
print("date.weekday() du 31 juillet 2026 :", date(2026, 7, 31).weekday(), "==", int(FRI))
print("un week-end :", int(SAT), int(SUN))

valeurs : {'MON': 0, 'TUE': 1, 'WED': 2, 'THU': 3, 'FRI': 4, 'SAT': 5, 'SUN': 6}
date.weekday() du 31 juillet 2026 : 4 == 4
un week-end : 5 6


In [3]:
# Elles marchent aussi sur des calendriers dont la semaine n'est pas lundi-vendredi.
print("2e mardi de chaque trimestre :", iso(bcal.nth_weekday("2026-01-01", "2026-12-31", 2, TUE, freq="Q")))
print("derniers samedis du trimestre:", iso(bcal.last_weekday("2026-01-01", "2026-06-30", SAT, freq="Q")))

2e mardi de chaque trimestre : ['2026-01-13', '2026-04-14', '2026-07-14', '2026-10-13']
derniers samedis du trimestre: ['2026-03-28', '2026-06-27']


## 1. Les deux exemples fondateurs

In [4]:
derniers_vendredis = bcal.last_weekday("2026-01-01", "2026-12-31", FRI)
pd.DataFrame(
    {"date": iso(derniers_vendredis), "jour": derniers_vendredis.strftime("%A")}
).set_index("date")

,jour
date,
2026-01-30,Friday
2026-02-27,Friday
2026-03-27,Friday
2026-04-24,Friday
2026-05-29,Friday
2026-06-26,Friday
2026-07-31,Friday
2026-08-28,Friday
2026-09-25,Friday


In [5]:
deuxiemes_jeudis = bcal.nth_weekday("2026-01-01", "2026-12-31", 2, THU, freq="Q")
iso(deuxiemes_jeudis)

['2026-01-08', '2026-04-09', '2026-07-09', '2026-10-08']

## 2. Trois conventions à connaître

### `n` est 1-based, et négatif compte depuis la fin

C'est tout l'intérêt : « le dernier vendredi du mois » est ce qu'on demande réellement,
et le calculer à la main est précisément là où sont les bugs.

In [6]:
janvier = pd.DataFrame(
    [{"n": n, "résultat": iso(bcal.nth_weekday("2026-01-01", "2026-01-31", n, FRI))}
     for n in (1, 2, 3, 4, 5, -1, -2, -5)]
).set_index("n")
janvier

,résultat
n,
1,[2026-01-02]
2,[2026-01-09]
3,[2026-01-16]
4,[2026-01-23]
5,[2026-01-30]
-1,[2026-01-30]
-2,[2026-01-23]
-5,[2026-01-02]


### Une occurrence absente est **sautée silencieusement**

Février a rarement un cinquième vendredi. Lever ferait de l'API un outil inutilisable sur
n'importe quelle plage réelle, donc le résultat a simplement moins de lignes que de mois.

In [7]:
cinquiemes = bcal.nth_weekday("2026-01-01", "2026-12-31", 5, FRI)
print(f"{len(cinquiemes)} mois sur 12 ont un cinquième vendredi :")
print(iso(cinquiemes))

4 mois sur 12 ont un cinquième vendredi :
['2026-01-30', '2026-05-29', '2026-07-31', '2026-10-30']


### L'occurrence appartient à la **période**, pas à la fenêtre

« Le dernier vendredi de janvier » est une propriété de janvier. Interroger à partir du 15
renvoie quand même le 30 ; une occurrence qui tombe avant la fenêtre est filtrée.

In [8]:
print("depuis le 15 janvier, dernier vendredi :", iso(bcal.last_weekday("2026-01-15", "2026-01-31", FRI)))
print("depuis le 15 janvier, premier vendredi :", iso(bcal.nth_weekday("2026-01-15", "2026-01-31", 1, FRI)))

depuis le 15 janvier, dernier vendredi : ['2026-01-30']
depuis le 15 janvier, premier vendredi : []


## 3. Les autres récurrences

In [9]:
print("1er jour du mois        :", iso(bcal.nth_day("2026-01-01", "2026-03-31", 1)))
print("dernier jour du mois    :", iso(bcal.nth_day("2026-01-01", "2026-03-31", -1)))
print("1er jour ouvré NYSE     :", iso(bcal.nth_business_day("2026-01-01", "2026-03-31", 1, cal="XNYS")))
print("dernier ouvré NYSE      :", iso(bcal.nth_business_day("2026-01-01", "2026-03-31", -1, cal="XNYS")))

1er jour du mois        : ['2026-01-01', '2026-02-01', '2026-03-01']
dernier jour du mois    : ['2026-01-31', '2026-02-28', '2026-03-31']
1er jour ouvré NYSE     : ['2026-01-02', '2026-02-02', '2026-03-02']
dernier ouvré NYSE      : ['2026-01-30', '2026-02-27', '2026-03-31']


`month_ends` illustre un point de conception : passer `cal` change la **question**, pas
seulement la réponse. Sans calendrier, on obtient le dernier jour *calendaire* ; avec, le
dernier jour *ouvré*. Le 31 janvier et le 28 février 2026 sont des samedis.

In [10]:
pd.DataFrame(
    {
        "calendaire": iso(bcal.month_ends("2026-01-01", "2026-06-30")),
        "ouvré (NYSE)": iso(bcal.month_ends("2026-01-01", "2026-06-30", cal="XNYS")),
        "ouvré (TARGET2)": iso(bcal.month_ends("2026-01-01", "2026-06-30", cal="fin:TARGET2")),
    }
)

,calendaire,ouvré (NYSE),ouvré (TARGET2)
0,2026-01-31,2026-01-30,2026-01-30
1,2026-02-28,2026-02-27,2026-02-27
2,2026-03-31,2026-03-31,2026-03-31
3,2026-04-30,2026-04-30,2026-04-30
4,2026-05-31,2026-05-29,2026-05-29
5,2026-06-30,2026-06-30,2026-06-30


In [11]:
print("trimestres          :", iso(bcal.quarter_ends("2026-01-01", "2026-12-31")))
print("exercice fiscal fév :", iso(bcal.quarter_ends("2026-01-01", "2026-12-31", anchor_month=2)))
print("années              :", iso(bcal.year_ends("2025-01-01", "2027-06-30")))

trimestres          : ['2026-03-31', '2026-06-30', '2026-09-30', '2026-12-31']
exercice fiscal fév : ['2026-02-28', '2026-05-31', '2026-08-31', '2026-11-30']
années              : ['2025-12-31', '2026-12-31']


### IMM et expirations d'options

Les dates IMM sont le 3e mercredi de mars / juin / septembre / décembre — c'est-à-dire du
**dernier mois du trimestre**, pas du trimestre. La nuance compte :

In [12]:
print("3e mercredi du trimestre :", iso(bcal.nth_weekday("2026-01-01", "2026-03-31", 3, WED, freq="Q")))
print("IMM (3e mercredi du mois):", iso(bcal.imm_dates("2026-01-01", "2026-03-31")))
print()
print("IMM 2026-2027 :", iso(bcal.imm_dates("2026-01-01", "2027-12-31")))

3e mercredi du trimestre : ['2026-01-21']
IMM (3e mercredi du mois): ['2026-03-18']

IMM 2026-2027 : ['2026-03-18', '2026-06-17', '2026-09-16', '2026-12-16', '2027-03-17', '2027-06-16', '2027-09-15', '2027-12-15']


Les expirations d'options sont le 3e vendredi, ajusté **en arrière** — quand le 3e vendredi
est le Vendredi saint, l'expiration recule au jeudi. C'était le cas en avril 2022 :

In [13]:
pd.DataFrame(
    {
        "3e vendredi brut": iso(bcal.nth_weekday("2022-01-01", "2022-06-30", 3, FRI)),
        "expiration NYSE": iso(bcal.option_expiries("2022-01-01", "2022-06-30", cal="XNYS")),
    }
)

,3e vendredi brut,expiration NYSE
0,2022-01-21,2022-01-21
1,2022-02-18,2022-02-18
2,2022-03-18,2022-03-18
3,2022-04-15,2022-04-14
4,2022-05-20,2022-05-20
5,2022-06-17,2022-06-17


## 4. `Schedule` — et la règle qui structure tout

La génération se fait en **deux étapes strictement séparées** :

1. `unadjusted()` — arithmétique de dates pure. Aucun calendrier, aucun férié, aucun roll.
2. `dates()` — applique `cal.adjust` par-dessus.

Les deux ne sont **jamais** entrelacées. La raison est la réconciliation : un système
en aval qui détient un trade bookié l'an dernier doit pouvoir vérifier que son coupon du
15 mars est bien la même date contractuelle que la nôtre, même si un férié a déplacé la
date de paiement effective. Si l'échéancier non ajusté dépendait des données de fériés,
régénérer un snapshot ferait apparaître le **contrat** comme modifié.

In [14]:
echeancier = Schedule("2026-02-28", "2027-08-31", freq="6M", cal="XNYS", eom=True)

pd.DataFrame(
    {
        "non ajusté (contractuel)": iso(echeancier.unadjusted()),
        "ajusté (paiement réel)": iso(echeancier.dates()),
    }
)

,non ajusté (contractuel),ajusté (paiement réel)
0,2026-02-28,2026-02-27
1,2026-08-31,2026-08-31
2,2027-02-28,2027-02-26
3,2027-08-31,2027-08-31


Démontrons la propriété : le non ajusté est **identique** quel que soit le calendrier,
même avec un férié posé pile sur une date de coupon. Seul l'ajusté bouge.

In [15]:
from better_calendar import Calendar

calendriers = {
    "aucun (weekday)": None,
    "XNYS": "XNYS",
    "TARGET2": "fin:TARGET2",
    "férié sur le coupon": Calendar("odd", holidays=["2026-08-31", "2026-09-01"]),
}

lignes = {}
for etiquette, cal in calendriers.items():
    s = Schedule("2026-02-28", "2027-08-31", freq="6M", cal=cal, eom=True)
    lignes[etiquette] = {"non ajusté": iso(s.unadjusted()), "ajusté": iso(s.dates())}
pd.DataFrame(lignes).T

,non ajusté,ajusté
aucun (weekday),"[2026-02-28, 2026-08-31, 2027-02-28, 2027-08-31]","[2026-02-27, 2026-08-31, 2027-02-26, 2027-08-31]"
XNYS,"[2026-02-28, 2026-08-31, 2027-02-28, 2027-08-31]","[2026-02-27, 2026-08-31, 2027-02-26, 2027-08-31]"
TARGET2,"[2026-02-28, 2026-08-31, 2027-02-28, 2027-08-31]","[2026-02-27, 2026-08-31, 2027-02-26, 2027-08-31]"
férié sur le coupon,"[2026-02-28, 2026-08-31, 2027-02-28, 2027-08-31]","[2026-02-27, 2026-08-28, 2027-02-26, 2027-08-31]"


### Les stubs

Un échéancier tombe rarement juste. Le *stub* est ce qu'on fait du reste. Un terme de cinq
mois à fréquence trimestrielle laisse deux mois à placer :

In [16]:
pd.DataFrame(
    [
        {"stub": stub, "dates": iso(Schedule("2026-01-15", "2026-06-15", freq="3M", stub=stub).unadjusted())}
        for stub in ("short_front", "long_front", "short_back", "long_back")
    ]
).set_index("stub")

,dates
stub,
short_front,"[2026-01-15, 2026-03-15, 2026-06-15]"
long_front,"[2026-01-15, 2026-06-15]"
short_back,"[2026-01-15, 2026-04-15, 2026-06-15]"
long_back,"[2026-01-15, 2026-06-15]"


Le choix du stub détermine **de quel bout** la grille régulière est mesurée : un stub
*front* ancre sur la date de fin et remonte, un stub *back* ancre sur le début et descend.
C'est ce qui fait que les coupons tombent sur la maturité au lieu de dériver.

In [17]:
front = Schedule("2026-01-10", "2027-01-15", freq="6M", stub="short_front")
back = Schedule("2026-01-10", "2027-01-15", freq="6M", stub="short_back")
print("front (ancré sur la fin)   :", iso(front.unadjusted()))
print("back  (ancré sur le début) :", iso(back.unadjusted()))

front (ancré sur la fin)   : ['2026-01-10', '2026-01-15', '2026-07-15', '2027-01-15']
back  (ancré sur le début) : ['2026-01-10', '2026-07-10', '2027-01-10', '2027-01-15']


In [18]:
# `none` refuse un terme qui ne tombe pas juste, au lieu d'inventer un stub.
try:
    Schedule("2026-01-15", "2026-06-15", freq="3M", stub="none").unadjusted()
except bcal.ScheduleError as exc:
    print(exc)

2026-01-15 to 2026-06-15 is not a whole number of 3M periods, and stub='none' forbids a stub. Choose a stub convention, or move one of the dates.


### Les dates sont mesurées depuis l'ancre, jamais pas à pas

Sinon le 31 janvier glisserait au 28 février puis **resterait** sur le 28 pour la vie du
trade. Ici, il revient au 31 dès que le mois le permet :

In [19]:
iso(Schedule("2026-01-31", "2026-06-30", freq="1M", stub="short_back").unadjusted())

['2026-01-31',
 '2026-02-28',
 '2026-03-31',
 '2026-04-30',
 '2026-05-31',
 '2026-06-30']

### Périodes d'accrual

In [20]:
echeancier = Schedule("2026-01-15", "2027-01-15", freq="3M", cal="XNYS")
pd.DataFrame(
    [
        {
            "début": p.start,
            "fin": p.end,
            "jours calendaires": len(p),
            "jours ouvrés": len(p.business_days("XNYS")),
        }
        for p in echeancier.periods()
    ]
)

,début,fin,jours calendaires,jours ouvrés
0,2026-01-15,2026-04-15,90,61
1,2026-04-15,2026-07-15,91,62
2,2026-07-15,2026-10-15,92,65
3,2026-10-15,2027-01-15,92,63


## 5. Un cas complet : coupons d'une obligation

Semestriel, du 15 mars 2026 au 15 mars 2031, réglé en zone euro, avec le lag de règlement
appliqué à chaque date de paiement.

In [21]:
obligation = Schedule("2026-03-15", "2031-03-15", freq="6M", cal="fin:TARGET2", eom=False)

coupons = pd.DataFrame(
    {
        "contractuel": iso(obligation.unadjusted()),
        "date de paiement": iso(obligation.dates()),
    }
)
coupons["décalé ?"] = coupons["contractuel"] != coupons["date de paiement"]
coupons["jours ouvrés depuis le précédent"] = [None] + [
    bcal.count(a, b, cal="fin:TARGET2")
    for a, b in zip(coupons["date de paiement"], coupons["date de paiement"][1:])
]
coupons

,contractuel,date de paiement,décalé ?,jours ouvrés depuis le précédent
0,2026-03-15,2026-03-16,True,NaN
1,2026-09-15,2026-09-15,False,128.0
2,2027-03-15,2027-03-15,False,127.0
3,2027-09-15,2027-09-15,False,130.0
4,2028-03-15,2028-03-15,False,130.0
5,2028-09-15,2028-09-15,False,129.0
6,2029-03-15,2029-03-15,False,126.0
7,2029-09-15,2029-09-17,True,129.0
8,2030-03-15,2030-03-15,False,126.0
9,2030-09-15,2030-09-16,True,128.0


## Récapitulatif

| Appel | Rôle |
|---|---|
| `nth_weekday(start, end, n, jour, freq=)` | n-ième jour de semaine de chaque période |
| `last_weekday(...)` | idem avec `n=-1` |
| `nth_day`, `nth_business_day` | n-ième jour calendaire / ouvré |
| `month_ends`, `quarter_ends`, `year_ends` | fins de période, calendaires ou ouvrées |
| `imm_dates`, `option_expiries` | les récurrences qui ont un nom |
| `Schedule(...).unadjusted()` | dates contractuelles, sans calendrier |
| `Schedule(...).dates()` | dates de paiement, ajustées |
| `Schedule(...).periods()` | périodes d'accrual |
| `MON`…`SUN` | constantes alignées sur `date.weekday()` |

**Suite :** [05 — Fuseaux et sessions](05-fuseaux-et-sessions.ipynb)